In [8]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-chroma
!pip install pypdf
!pip install langchain-text-splitters

In [2]:
from getpass import getpass
import os

OPENAI_KEY = getpass()

os.environ["OPENAI_API_KEY"] = OPENAI_KEY

 ········


In [4]:
!pip install langchain-community

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 625.4 kB/s eta 0:00:03
   -------- ------------------------------- 0.5/2.4 MB 625.4 kB/s eta 0:00:03
   -------- ------------------------------- 0.5/2.4 MB 625.4 kB/s eta 0:00:03
   ------------- -------------------------- 0.8/2.4 MB 436.0 kB/s eta 0:00:04
   ------------- -------------------------- 0.8/2.4 MB 436.0 kB/s eta 0:00:04
   ------------- -------------------------- 0.8/2.4 MB 436.0 kB/s eta 0:00:04
   ----------------- ---------------------- 1.0/2.4 MB 443.5 kB/s eta 0:00:03
   ----------------- ---------------------- 1.0/2.4 MB 443.5 kB/s eta 0:00:03
   ---------------------- ----------------- 1.3/2.4 MB 479.3 kB/s eta 0:00:03
   -------------------

In [9]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "AI_Growth_50_Page_Report.pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()

print("Pages:", len(docs))

Pages: 100


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunked_docs = splitter.split_documents(docs)

print("Chunks:", len(chunked_docs))

Chunks: 600


In [12]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [13]:
from langchain_chroma import Chroma

vectordb = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embedding_model,
    collection_name="pdf_rag"
)

In [14]:
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k":10}
)

In [15]:
from langchain_core.prompts import ChatPromptTemplate

prompt = """
You are an expert document summarizer.

Using the retrieved context, generate a concise summary.

Context:
{context}

Summary:
"""

prompt_template = ChatPromptTemplate.from_template(prompt)

In [16]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [17]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

In [18]:
from langchain_core.runnables import RunnablePassthrough

summary_chain = (
    {
        "context": retriever | format_docs
    }
    | prompt_template
    | llm
)

In [19]:
query = "Summarize the entire PDF"

result = summary_chain.invoke(query)

print(result.content)

The section on the growth of Artificial Intelligence (AI) outlines its evolution from a focus on symbolic reasoning to a foundational technology impacting various industries, including software, finance, healthcare, education, manufacturing, and scientific discovery. Initially, AI systems depended on handcrafted rules and expert knowledge. However, advancements in computing power and the availability of larger datasets led to the rise of machine learning techniques, which began to surpass rule-based methods. The advent of deep learning further accelerated progress, resulting in significant breakthroughs in areas such as computer vision, natural language processing, speech recognition, and recommendation systems. Transformer architectures have since revolutionized the field by enabling models to understand complex relationships within extensive data sets. The discussion also highlights trends, innovation, industry adoption, governance, research advancements, and future opportunities in 

In [20]:
retriever = vectordb.as_retriever(
    search_kwargs={"k":25}
)

In [21]:
query = """
Provide:
1. Executive Summary
2. Key Topics
3. Important Findings
4. Conclusions
"""

In [22]:
print(query)


Provide:
1. Executive Summary
2. Key Topics
3. Important Findings
4. Conclusions

